### 0️. Importar Librerias

In [35]:
import sys
from pathlib import Path

sys.path.append(str(Path().resolve().parent))

In [36]:
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import (
    train_test_split,
    GridSearchCV
)

from sklearn.preprocessing import (
    StandardScaler
)

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,

    accuracy_score,
    precision_score,
    recall_score,
    f1_score,

    confusion_matrix,
    classification_report
)

from sklearn.linear_model import (
    LinearRegression, 
    LogisticRegression, 
    Ridge, 
    Lasso, 
    ElasticNet)

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier


# Para que se muestren todas las columnas al inspeccionar los DataFrames
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 2000)
pd.set_option('display.expand_frame_repr', False)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', None)

In [37]:
from framework import sp_carga_exploracion as sc
from framework import sp_limpieza_transformacion as sl
from framework import sp_eda as se
from framework import sp_abtest as sb
from framework import sp_modelado as sm

### 1️. Cargar DataFrame 

#### DataFrame 

In [38]:
ruta = (r"../data/processed/03_datos_eda.csv")
df_md = sc.leer_csv(ruta)

CSV CARGADO
Archivo: ../data/processed/03_datos_eda.csv
Filas:   1000
Columnas: 14


####  escuela copia del df_md

In [39]:
escuela = df_md.copy()

### 2. Análisis de Datos

#### clasificar_columnas()

In [40]:
sm.clasificar_columnas(escuela)

,columna,dtype,n_unicos,sugerencia,motivo
0,tiene_tutor_sn,str,2,col_control,Categórica con 2 valores únicos: rango razonable para agrupar
1,aprobado_sn,str,2,col_control,Categórica con 2 valores únicos: rango razonable para agrupar
2,nivel_dificultad,str,3,col_control,Categórica con 3 valores únicos: rango razonable para agrupar
3,horario_estudio_preferido,str,4,col_control,Categórica con 4 valores únicos: rango razonable para agrupar
4,estilo_aprendizaje,str,5,col_control,Categórica con 5 valores únicos: rango razonable para agrupar
5,rendimiento_academico,str,5,col_control,Categórica con 5 valores únicos: rango razonable para agrupar
6,nota_final,float64,364,metrica,Numérica con 364 valores únicos: variable continua medible
7,horas_sueno,float64,811,metrica,Numérica con 811 valores únicos: variable continua medible
8,tasa_asistencia,float64,893,metrica,Numérica con 893 valores únicos: variable continua medible
9,horas_estudio_semanal,float64,964,metrica,Numérica con 964 valores únicos: variable continua medible


In [41]:
escuela.columns.tolist()

['horas_estudio_semanal',
 'nota_anterior',
 'tasa_asistencia',
 'horas_sueno',
 'edad',
 'nivel_dificultad',
 'tiene_tutor_sn',
 'horario_estudio_preferido',
 'estilo_aprendizaje',
 'nota_final',
 'aprobado_ml',
 'aprobado_sn',
 'tiene_tutor_ml',
 'rendimiento_academico']

#### Regresión

##### target values Regresión

In [42]:
tv_reg = 'nota_final'

##### cols_excluir_reg

In [43]:
# Porque derivan directa o indirectamente de nota_final y son tv_clf.
cols_excluir_reg = ['aprobado_ml','aprobado_sn','rendimiento_academico','tiene_tutor_sn']

 ##### datasets especifico

In [44]:
escuela_reg = escuela.drop(columns=cols_excluir_reg,errors='ignore')
escuela_reg.sample(3)

,horas_estudio_semanal,nota_anterior,tasa_asistencia,horas_sueno,edad,nivel_dificultad,horario_estudio_preferido,estilo_aprendizaje,nota_final,tiene_tutor_ml
93,8.051083,78.288053,84.673159,7.007670,21,medio,desconocido,visual,80.4,0
0,8.957476,48.830601,86.640182,6.675694,25,facil,tarde,lectura_escritura,84.4,1
487,6.159623,76.091182,89.046308,7.039523,21,dificil,tarde,auditivo,71.5,0


##### predictorias 

In [45]:
predictoras_reg = escuela_reg.drop(columns=[tv_reg]).columns.tolist()
predictoras_reg 

['horas_estudio_semanal',
 'nota_anterior',
 'tasa_asistencia',
 'horas_sueno',
 'edad',
 'nivel_dificultad',
 'horario_estudio_preferido',
 'estilo_aprendizaje',
 'tiene_tutor_ml']

#### Clasificación

##### target values Clasificacion

In [46]:
tv_clf = 'aprobado_ml'

In [47]:
# Porque derivan directa o indirectamente de aprobado y nota final y son tv_reg
cols_excluir_clf = ['nota_final','aprobado_sn','rendimiento_academico','tiene_tutor_sn']

In [48]:
escuela_clf = escuela.drop(columns=cols_excluir_clf,errors='ignore')
escuela_clf.sample(3)

,horas_estudio_semanal,nota_anterior,tasa_asistencia,horas_sueno,edad,nivel_dificultad,horario_estudio_preferido,estilo_aprendizaje,aprobado_ml,tiene_tutor_ml
28,13.706938,66.795458,44.408836,5.587450,27,medio,desconocido,auditivo,1,1
333,12.704412,61.886763,73.467929,8.566613,18,medio,noche,kinestesico,1,1
502,6.924877,68.170074,75.276014,7.007670,22,facil,manana,visual,1,0


In [49]:
predictoras_clf = escuela_clf.drop(columns=[tv_clf]).columns.tolist()
predictoras_clf 

['horas_estudio_semanal',
 'nota_anterior',
 'tasa_asistencia',
 'horas_sueno',
 'edad',
 'nivel_dificultad',
 'horario_estudio_preferido',
 'estilo_aprendizaje',
 'tiene_tutor_ml']

## A. REGRESION 

### 3. Preparación de Datos

In [50]:
escuela_reg = sm.preparacion_dataframe(escuela_reg)
escuela_reg.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 10 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   horas_estudio_semanal      1000 non-null   float64
 1   nota_anterior              1000 non-null   float64
 2   tasa_asistencia            1000 non-null   float64
 3   horas_sueno                1000 non-null   float64
 4   edad                       1000 non-null   int64  
 5   nivel_dificultad           1000 non-null   str    
 6   horario_estudio_preferido  1000 non-null   str    
 7   estilo_aprendizaje         1000 non-null   str    
 8   nota_final                 1000 non-null   float64
 9   tiene_tutor_ml             1000 non-null   int64  
dtypes: float64(5), int64(2), str(3)
memory usage: 98.3 KB


### 4. Separar Variables

#### Variables para modelado

In [51]:
escuela_reg.sample(3)

,horas_estudio_semanal,nota_anterior,tasa_asistencia,horas_sueno,edad,nivel_dificultad,horario_estudio_preferido,estilo_aprendizaje,nota_final,tiene_tutor_ml
879,19.632016,75.402576,81.723557,7.00767,25,medio,tarde,visual,82.9,1
184,5.705452,66.020742,65.775708,7.00767,29,dificil,noche,visual,48.5,0
273,9.742700,74.096335,58.536568,7.00767,19,medio,manana,kinestesico,72.4,0


In [52]:
tv_reg

'nota_final'

In [53]:
X_reg, y_reg = sm.separar_xy(
    escuela_reg,
    tv_reg
)

In [54]:
print(X_reg.shape)
print(y_reg.shape)

display(X_reg.head())
display(y_reg.head())

(1000, 9)
(1000,)


,horas_estudio_semanal,nota_anterior,tasa_asistencia,horas_sueno,edad,nivel_dificultad,horario_estudio_preferido,estilo_aprendizaje,tiene_tutor_ml
0,8.957476,48.830601,86.640182,6.675694,25,facil,tarde,lectura_escritura,1
1,11.042524,80.825707,83.449655,4.616844,18,dificil,tarde,desconocido,0
2,4.510776,90.383694,74.623607,7.755246,25,facil,manana,lectura_escritura,0
3,6.647213,81.878257,82.849841,8.592826,23,facil,desconocido,visual,0
4,1.000000,66.254179,54.539935,6.671840,21,medio,desconocido,auditivo,0


0    84.4
1    72.0
2    80.0
3    78.2
4    66.0
Name: nota_final, dtype: float64

Conclusiones

Se separa la variable objetivo (`nota_final`) de las variables predictoras.

Resultado:

- Observaciones: 1.000
- Variables predictoras: 9
- Variable objetivo: nota_final

Los conjuntos quedan preparados para la fase de transformación de variables.

### 5. Transformación de Variables

#### nulos 

In [55]:
escuela_reg.isna().sum()

horas_estudio_semanal        0
nota_anterior                0
tasa_asistencia              0
horas_sueno                  0
edad                         0
nivel_dificultad             0
horario_estudio_preferido    0
estilo_aprendizaje           0
nota_final                   0
tiene_tutor_ml               0
dtype: int64

✓ No hay nulos.
✓ tiene_tutor_ml ya está codificada.
✓ No es necesario codificar variables binarias.
✓ Hay 3 variables categóricas que requieren One Hot Encoding.

#### variables_numericas / categoricas / binarias  

In [56]:
escuela_reg.dtypes

horas_estudio_semanal        float64
nota_anterior                float64
tasa_asistencia              float64
horas_sueno                  float64
edad                           int64
nivel_dificultad                 str
horario_estudio_preferido        str
estilo_aprendizaje               str
nota_final                   float64
tiene_tutor_ml                 int64
dtype: object

In [57]:
numericas_reg = [
    'horas_estudio_semanal',
    'nota_anterior',
    'tasa_asistencia',
    'horas_sueno',
    'edad',
]

categoricas_reg = [
    'nivel_dificultad',
    'horario_estudio_preferido',
    'estilo_aprendizaje'
]

binaria_ml = [
    'tiene_tutor_ml'
]

 #### tratar_numericas()

In [58]:
# Numéricas
X_reg = sm.tratar_numericas(
    X_reg,
    numericas_reg
)

#### tratar_categoricas()

In [59]:
# Categóricas
X_reg = sm.tratar_categoricas(
    X_reg,
    categoricas_reg
)



#### codificar_categoricas()

In [60]:
# One Hot Encoding
X_reg = sm.codificar_categoricas(
    X_reg,
    categoricas_reg
)

In [61]:
print(X_reg.shape)

display(X_reg.head())

(1000, 18)


,horas_estudio_semanal,nota_anterior,tasa_asistencia,horas_sueno,edad,tiene_tutor_ml,nivel_dificultad_dificil,nivel_dificultad_facil,nivel_dificultad_medio,horario_estudio_preferido_desconocido,horario_estudio_preferido_manana,horario_estudio_preferido_noche,horario_estudio_preferido_tarde,estilo_aprendizaje_auditivo,estilo_aprendizaje_desconocido,estilo_aprendizaje_kinestesico,estilo_aprendizaje_lectura_escritura,estilo_aprendizaje_visual
0,8.957476,48.830601,86.640182,6.675694,25,1,0,1,0,0,0,0,1,0,0,0,1,0
1,11.042524,80.825707,83.449655,4.616844,18,0,1,0,0,0,0,0,1,0,1,0,0,0
2,4.510776,90.383694,74.623607,7.755246,25,0,0,1,0,0,1,0,0,0,0,0,1,0
3,6.647213,81.878257,82.849841,8.592826,23,0,0,1,0,1,0,0,0,0,0,0,0,1
4,1.000000,66.254179,54.539935,6.671840,21,0,0,0,1,1,0,0,0,1,0,0,0,0


In [62]:
X_reg.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 18 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   horas_estudio_semanal                  1000 non-null   float64
 1   nota_anterior                          1000 non-null   float64
 2   tasa_asistencia                        1000 non-null   float64
 3   horas_sueno                            1000 non-null   float64
 4   edad                                   1000 non-null   int64  
 5   tiene_tutor_ml                         1000 non-null   int64  
 6   nivel_dificultad_dificil               1000 non-null   int64  
 7   nivel_dificultad_facil                 1000 non-null   int64  
 8   nivel_dificultad_medio                 1000 non-null   int64  
 9   horario_estudio_preferido_desconocido  1000 non-null   int64  
 10  horario_estudio_preferido_manana       1000 non-null   int64  
 11  horario_estudio_

#### Conclusiones

Se han transformado correctamente las variables predictoras:

- Variables numéricas validadas.
- Variables categóricas codificadas mediante One Hot Encoding.
- Variables binarias conservadas en formato 0/1.
- No existen valores nulos.
- Todas las variables predictoras son numéricas y compatibles con algoritmos de Machine Learning.

Dataset final de modelado:

- Observaciones: 1.000
- Variables predictoras: 18

### Guardar archivos csv Regresión

#### guardar_csv X_reg

In [63]:
X_reg.sample()

,horas_estudio_semanal,nota_anterior,tasa_asistencia,horas_sueno,edad,tiene_tutor_ml,nivel_dificultad_dificil,nivel_dificultad_facil,nivel_dificultad_medio,horario_estudio_preferido_desconocido,horario_estudio_preferido_manana,horario_estudio_preferido_noche,horario_estudio_preferido_tarde,estilo_aprendizaje_auditivo,estilo_aprendizaje_desconocido,estilo_aprendizaje_kinestesico,estilo_aprendizaje_lectura_escritura,estilo_aprendizaje_visual
278,3.508645,63.539091,100.0,7.114905,24,0,1,0,0,0,0,0,1,1,0,0,0,0


In [64]:
ruta = '../data/processed/X_reg.csv'
sm.guardar_csv(X_reg,ruta)

ARCHIVO GUARDADO
Ruta:     ../data/processed/X_reg.csv
Filas:    1000
Columnas: 18


#### guardar_csv y_reg

In [65]:
y_reg.sample()

411    73.6
Name: nota_final, dtype: float64

In [66]:
ruta = '../data/processed/y_reg.csv'
sm.guardar_csv(y_reg,ruta)

ARCHIVO GUARDADO
Ruta:     ../data/processed/y_reg.csv
Filas:    1000


## B. CLASIFICACION

### 3. Preparación de Datos

In [67]:
escuela_cfl = sm.preparacion_dataframe(escuela_clf)
escuela_clf.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 10 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   horas_estudio_semanal      1000 non-null   float64
 1   nota_anterior              1000 non-null   float64
 2   tasa_asistencia            1000 non-null   float64
 3   horas_sueno                1000 non-null   float64
 4   edad                       1000 non-null   int64  
 5   nivel_dificultad           1000 non-null   str    
 6   horario_estudio_preferido  1000 non-null   str    
 7   estilo_aprendizaje         1000 non-null   str    
 8   aprobado_ml                1000 non-null   int64  
 9   tiene_tutor_ml             1000 non-null   int64  
dtypes: float64(4), int64(3), str(3)
memory usage: 98.3 KB


### 4. Separar Variables

#### Variables para modelado

In [68]:
escuela_clf.sample(3)

,horas_estudio_semanal,nota_anterior,tasa_asistencia,horas_sueno,edad,nivel_dificultad,horario_estudio_preferido,estilo_aprendizaje,aprobado_ml,tiene_tutor_ml
313,6.488423,88.152059,78.200208,6.775221,26,medio,tarde,desconocido,1,0
434,5.311011,54.718328,58.677002,6.737082,23,medio,tarde,visual,1,0
747,7.382315,71.867850,71.341449,7.708610,19,dificil,noche,kinestesico,1,0


In [69]:
tv_clf

'aprobado_ml'

In [70]:
X_clf, y_clf = sm.separar_xy(
    escuela_clf,
    tv_clf
)

In [71]:
print(X_clf.shape)
print(y_clf.shape)

display(X_clf.head())
display(y_clf.head())

(1000, 9)
(1000,)


,horas_estudio_semanal,nota_anterior,tasa_asistencia,horas_sueno,edad,nivel_dificultad,horario_estudio_preferido,estilo_aprendizaje,tiene_tutor_ml
0,8.957476,48.830601,86.640182,6.675694,25,facil,tarde,lectura_escritura,1
1,11.042524,80.825707,83.449655,4.616844,18,dificil,tarde,desconocido,0
2,4.510776,90.383694,74.623607,7.755246,25,facil,manana,lectura_escritura,0
3,6.647213,81.878257,82.849841,8.592826,23,facil,desconocido,visual,0
4,1.000000,66.254179,54.539935,6.671840,21,medio,desconocido,auditivo,0


0    1
1    1
2    1
3    1
4    1
Name: aprobado_ml, dtype: int64

Conclusiones

Se separa la variable objetivo (`aprobado_ml`) de las variables predictoras.

Resultado:

- Observaciones: 1.000
- Variables predictoras: 9
- Variable objetivo: nota_final

Los conjuntos quedan preparados para la fase de transformación de variables.

### 5. Transformación de Variables

#### nulos 

In [73]:
escuela_clf.isna().sum()

horas_estudio_semanal        0
nota_anterior                0
tasa_asistencia              0
horas_sueno                  0
edad                         0
nivel_dificultad             0
horario_estudio_preferido    0
estilo_aprendizaje           0
aprobado_ml                  0
tiene_tutor_ml               0
dtype: int64

✓ No hay nulos.
✓ aprobado_ml ya está codificada.
✓ No es necesario codificar variables binarias.
✓ Hay 3 variables categóricas que requieren One Hot Encoding.

#### variables_numericas / categoricas / binarias  

In [72]:
escuela_clf.dtypes

horas_estudio_semanal        float64
nota_anterior                float64
tasa_asistencia              float64
horas_sueno                  float64
edad                           int64
nivel_dificultad                 str
horario_estudio_preferido        str
estilo_aprendizaje               str
aprobado_ml                    int64
tiene_tutor_ml                 int64
dtype: object

In [75]:
numericas_clf = [
    'horas_estudio_semanal',
    'nota_anterior',
    'tasa_asistencia',
    'horas_sueno',
    'edad'
]

categoricas_clf = [
    'nivel_dificultad',
    'horario_estudio_preferido',
    'estilo_aprendizaje'
]

binaria_ml = [
    'tiene_tutor_ml'
]

 #### tratar_numericas()

In [77]:
X_clf = sm.tratar_numericas(
    X_clf,
    numericas_clf
)

#### tratar_categoricas()

In [78]:
X_clf = sm.tratar_categoricas(
    X_clf,
    categoricas_clf
)

#### codificar_categoricas()

In [79]:
X_clf = sm.codificar_categoricas(
    X_clf,
    categoricas_clf
)

#### Conclusiones

In [80]:
print(X_clf.shape)

display(X_clf.head())

(1000, 18)


,horas_estudio_semanal,nota_anterior,tasa_asistencia,horas_sueno,edad,tiene_tutor_ml,nivel_dificultad_dificil,nivel_dificultad_facil,nivel_dificultad_medio,horario_estudio_preferido_desconocido,horario_estudio_preferido_manana,horario_estudio_preferido_noche,horario_estudio_preferido_tarde,estilo_aprendizaje_auditivo,estilo_aprendizaje_desconocido,estilo_aprendizaje_kinestesico,estilo_aprendizaje_lectura_escritura,estilo_aprendizaje_visual
0,8.957476,48.830601,86.640182,6.675694,25,1,0,1,0,0,0,0,1,0,0,0,1,0
1,11.042524,80.825707,83.449655,4.616844,18,0,1,0,0,0,0,0,1,0,1,0,0,0
2,4.510776,90.383694,74.623607,7.755246,25,0,0,1,0,0,1,0,0,0,0,0,1,0
3,6.647213,81.878257,82.849841,8.592826,23,0,0,1,0,1,0,0,0,0,0,0,0,1
4,1.000000,66.254179,54.539935,6.671840,21,0,0,0,1,1,0,0,0,1,0,0,0,0


In [81]:
X_clf.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 18 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   horas_estudio_semanal                  1000 non-null   float64
 1   nota_anterior                          1000 non-null   float64
 2   tasa_asistencia                        1000 non-null   float64
 3   horas_sueno                            1000 non-null   float64
 4   edad                                   1000 non-null   int64  
 5   tiene_tutor_ml                         1000 non-null   int64  
 6   nivel_dificultad_dificil               1000 non-null   int64  
 7   nivel_dificultad_facil                 1000 non-null   int64  
 8   nivel_dificultad_medio                 1000 non-null   int64  
 9   horario_estudio_preferido_desconocido  1000 non-null   int64  
 10  horario_estudio_preferido_manana       1000 non-null   int64  
 11  horario_estudio_

### Guardar archivos csv clasificacion

#### guardar_csv X_clf 

In [82]:
X_clf.sample()

,horas_estudio_semanal,nota_anterior,tasa_asistencia,horas_sueno,edad,tiene_tutor_ml,nivel_dificultad_dificil,nivel_dificultad_facil,nivel_dificultad_medio,horario_estudio_preferido_desconocido,horario_estudio_preferido_manana,horario_estudio_preferido_noche,horario_estudio_preferido_tarde,estilo_aprendizaje_auditivo,estilo_aprendizaje_desconocido,estilo_aprendizaje_kinestesico,estilo_aprendizaje_lectura_escritura,estilo_aprendizaje_visual
710,11.300447,46.559178,99.95011,9.294337,29,0,0,0,1,0,0,1,0,0,0,1,0,0


In [83]:
ruta = '../data/processed/X_clf.csv'
sm.guardar_csv(X_clf,ruta)

ARCHIVO GUARDADO
Ruta:     ../data/processed/X_clf.csv
Filas:    1000
Columnas: 18


#### guardar_csv y_reg

In [84]:
y_clf.sample()

780    1
Name: aprobado_ml, dtype: int64

In [85]:
ruta = '../data/processed/y_clf.csv'
sm.guardar_csv(y_clf,ruta)

ARCHIVO GUARDADO
Ruta:     ../data/processed/y_clf.csv
Filas:    1000
